# Aula 2 - Introdução aos dados tabulares

## Retomando o projeto
Antes de começar, confiro onde parei: o histórico do Git e o estado da área de trabalho.

In [ ]:
%cd "C:/Users/USER/Documents/Projetos/Estudos/Ciencia de dados"

# onde paramos? o log mostra os commits ate aqui
!git log --oneline

# e o estado atual?
!git status

## Instalando a dependência para ler Excel
O pandas precisa da biblioteca openpyxl para ler arquivos .xlsx.
Instalo no ambiente do projeto (.venv), atualizo o requirements.txt e faço o commit.

In [ ]:
# instala dentro do .venv do projeto
!uv pip install --python .venv/Scripts/python.exe openpyxl

# o kernel do notebook usa o Python global, entao ele tambem precisa da biblioteca
%pip install openpyxl

# atualiza a receita com as versoes exatas
!uv pip freeze --color never --python .venv/Scripts/python.exe > requirements.txt
!type requirements.txt

In [ ]:
!git add requirements.txt
!git commit -m "Adiciona openpyxl para leitura de arquivos Excel"

## Pandas em escala pequena
Primeiro uma tabela pequena, em que dá para conferir cada valor a olho.
- Series: uma coluna (valores + índice)
- DataFrame: a tabela inteira (várias Series com o mesmo índice)

In [ ]:
import pandas as pd

# DataFrame construido a mao: cinco vendas ficticias
vendas = pd.DataFrame({
    "produto": ["Caneca", "Camiseta", "Caneca", "Adesivo", "Camiseta"],
    "quantidade": [2, 1, 4, 10, 3],
    "preco_unitario": [35.0, 79.9, 35.0, 5.5, 79.9],
    "pais": ["Brasil", "Brasil", "Portugal", "Brasil", "Argentina"],
})

vendas

Cada linha é uma observação (uma venda) e cada coluna é uma variável.
- `produto` e `pais`: categóricas nominais
- `quantidade`: numérica discreta
- `preco_unitario`: numérica contínua

In [ ]:
# 1. selecionar uma coluna: o resultado e uma Series
vendas["preco_unitario"]

In [ ]:
# 2. criar uma coluna nova a partir das existentes
vendas["faturamento"] = vendas["quantidade"] * vendas["preco_unitario"]
vendas

In [ ]:
# 3. filtrar linhas por uma condicao (filtro booleano)
# a condicao gera True/False para cada linha e o DataFrame devolve so as True
vendas[vendas["pais"] == "Brasil"]

In [ ]:
# varias condicoes: use & (e) e | (ou), cada uma entre parenteses
# as palavras and/or do Python nao funcionam com Series
vendas[(vendas["pais"] == "Brasil") & (vendas["faturamento"] > 50)]

In [ ]:
# describe: estatisticas descritivas das colunas numericas
vendas.describe()

## O dataset: Online Retail
Vendas de um varejo online do Reino Unido entre dez/2010 e dez/2011 (541.909 registros, cada linha é um item vendido em uma nota fiscal).
Fonte: UCI Machine Learning Repository (id 352), licença CC BY 4.0. Citação: Chen, D. (2015). Online Retail. UCI.

| Variável | Descrição | Tipo |
|---|---|---|
| InvoiceNo | Número da nota fiscal (começando com C é cancelamento) | categórica nominal |
| StockCode | Código do produto | categórica nominal |
| Description | Nome do produto | categórica nominal |
| Quantity | Quantidade do item na transação | numérica discreta |
| InvoiceDate | Data e hora da transação | temporal |
| UnitPrice | Preço unitário, em libras | numérica contínua |
| CustomerID | Código do cliente | categórica nominal |
| Country | País do cliente | categórica nominal |

### Baixando os dados
Os dados ficam na pasta `data/`, que não deve ir para o GitHub (arquivo grande, dado bruto).
Por isso primeiro coloco `data/` no .gitignore.

In [ ]:
from pathlib import Path

# acrescenta data/ no .gitignore so se ainda nao estiver la
gitignore = Path(".gitignore")
texto = gitignore.read_text()
if "data/" not in texto:
    gitignore.write_text(texto + "data/\n")

!type .gitignore

In [ ]:
pasta_dados = Path("Unidade_01/data")
pasta_dados.mkdir(exist_ok=True)

# baixa o arquivo compactado (cerca de 23 MB)
!curl -L -o Unidade_01/data/online_retail.zip "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

In [ ]:
import zipfile

# o Windows nao tem unzip no terminal, entao descompacto com o Python
with zipfile.ZipFile(pasta_dados / "online_retail.zip") as zip_dados:
    zip_dados.extractall(pasta_dados)

# renomeia para um nome sem espacos (replace sobrescreve se ja existir)
(pasta_dados / "Online Retail.xlsx").replace(pasta_dados / "online_retail.xlsx")

!dir Unidade_01\data

In [ ]:
# os arquivos novos aparecem no git status? (nao deveriam, data/ esta no .gitignore)
!git status

Os dados não vão para o repositório, então o README precisa ensinar como obtê-los.

In [ ]:
# documenta a fonte dos dados no README (so se ainda nao estiver la)
readme = Path("README.md")
texto = readme.read_text(encoding="utf-8")
if "## Dados" not in texto:
    secao = (
        "\n## Dados\n\n"
        "Dataset Online Retail (UCI Machine Learning Repository, id 352).\n"
        "Licenca CC BY 4.0. Citacao: Chen, D. (2015). Online Retail. UCI.\n"
        "Download: https://archive.ics.uci.edu/dataset/352/online+retail\n"
        "Apos baixar, descompacte em Unidade_01/data/ e renomeie para online_retail.xlsx\n"
    )
    readme.write_text(texto + secao, encoding="utf-8")

!type README.md

In [ ]:
!git add README.md .gitignore
!git commit -m "Documenta a fonte dos dados e ignora a pasta data"

## Ritual de inspeção inicial
Leitura do Excel. Atenção: arquivo grande, a célula pode levar um ou dois minutos.

In [ ]:
caminho = pasta_dados / "online_retail.xlsx"

# paciencia: arquivos Excel grandes sao lentos de ler
retail = pd.read_excel(caminho)

### Passo 1: olhar os dados

In [ ]:
# primeiras linhas
retail.head()

In [ ]:
# ultimas linhas (fim de arquivo pode esconder totais e lixo)
retail.tail()

### Passo 2: tamanho e tipos

In [ ]:
# quantas linhas e colunas?
retail.shape

In [ ]:
# tipos, valores nao nulos e memoria
retail.info()

Olhando o `info()`: `CustomerID` foi lido como float64. Isso acontece porque a coluna tem valores ausentes
(NaN é um número decimal e o tipo inteiro do pandas não comporta NaN, então tudo vira float).

### Passo 3: valores ausentes

In [ ]:
# quantos ausentes por coluna?
# isna() marca True onde falta valor e sum() soma os True de cada coluna
retail.isna().sum()

### Passo 4: resumo das variáveis numéricas

In [ ]:
retail.describe()

### Passo 5: contagem das categorias

In [ ]:
# as 10 maiores origens das vendas
retail["Country"].value_counts().head(10)

In [ ]:
# quantos paises distintos?
retail["Country"].nunique()

## Seleção e filtragem
Mesmas ferramentas da tabela pequena, agora no dataset real.

In [ ]:
# vendas para a Alemanha
vendas_alemanha = retail[retail["Country"] == "Germany"]
vendas_alemanha.shape

In [ ]:
# transacoes com preco unitario acima de 100 libras fora do Reino Unido
# filtra as linhas e escolhe so as colunas relevantes (lista de nomes entre colchetes)
caras_fora_uk = retail[(retail["UnitPrice"] > 100) & (retail["Country"] != "United Kingdom")]
caras_fora_uk[["InvoiceNo", "Description", "UnitPrice", "Country"]].head()

In [ ]:
# loc seleciona por rotulo/condicao, iloc por posicao
print(retail.loc[retail["Country"] == "France", ["InvoiceNo", "UnitPrice"]].head())
print(retail.iloc[0:5, 0:3])

### Perguntas de negócio
1. Quantas transações registram o país como Brazil?
2. Quantas linhas têm Quantity negativa? Como elas são?
3. Existe linha com UnitPrice igual a zero e CustomerID ausente ao mesmo tempo?

In [ ]:
# 1. transacoes do Brasil
brasil = retail[retail["Country"] == "Brazil"]
print(len(brasil), "transacoes do Brazil")

In [ ]:
# 2. quantidade negativa
negativas = retail[retail["Quantity"] < 0]
print(len(negativas), "linhas com quantidade negativa")
negativas.head()

In [ ]:
# 3. preco zero E cliente ausente ao mesmo tempo
zero_sem_cliente = retail[(retail["UnitPrice"] == 0) & (retail["CustomerID"].isna())]
print(len(zero_sem_cliente), "linhas com preco zero e sem CustomerID")

## Diagnóstico de qualidade
Aqui só diagnostico os problemas, sem tratar nada. A decisão do que fazer com cada um fica para a próxima aula.

### Suspeita 1: quantidades negativas

In [ ]:
# as notas das quantidades negativas comecam com C (cancelamento)?
notas_negativas = retail.loc[retail["Quantity"] < 0, "InvoiceNo"].astype(str)
print("Comecam com C:", notas_negativas.str.startswith("C").sum())
print("Total de negativas:", len(notas_negativas))

### Suspeita 2: preços zerados

In [ ]:
preco_zero = retail[retail["UnitPrice"] == 0]
print(len(preco_zero), "linhas com preco zero")
preco_zero[["InvoiceNo", "Description", "Quantity", "CustomerID"]].head()

### Suspeita 3: registros duplicados

In [ ]:
# linhas identicas em todas as colunas
print(retail.duplicated().sum(), "linhas duplicadas")

## Ficha de reconhecimento do dataset
(preencher depois de rodar o notebook: tamanho, tipos, ausentes e os três problemas de qualidade)

In [ ]:
# commit que encerra a sessao (salve o notebook antes)
!git add Unidade_01/aula_pratica_01/exercicio_aula02.ipynb
!git commit -m "Adiciona inspecao inicial e diagnostico do Online Retail"
!git log --oneline